# 01 — Data Audit & Dataset Profiling

EOR Atlas research-layer notebook. Run this notebook from VS Code/Jupyter.


In [5]:

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
import hashlib
import json
import re

import numpy as np
import pandas as pd


SEED = 42

# Locate the repository root robustly from a notebook working directory.
def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / "src").exists() and (p / "outputs").exists():
            return p
    # Fallback for running the notebook from src/notebooks.
    return Path.cwd().parents[2]


PROJECT_ROOT = find_project_root()
ML_DATA_DIR = PROJECT_ROOT / "src" / "notebooks" / "ml_data"
ARTIFACT_DIR = PROJECT_ROOT / "outputs" / "model_artifacts"
PROCESSED_DIR = PROJECT_ROOT / "outputs" / "ml_research"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


COLUMN_ALIASES = {
    "technique": [
        "EOR technique", "EOR Technique", "technique",
        "Technique", "EOR_Technique", "EOR Method", "Method"
    ],
    "formation": [
        "Formation type", "formation_category", "Formation",
        "Formation Type", "Rock Type", "rock_type"
    ],
    "depth_min": ["Depth min (ft)", "depth_min_ft", "Depth_min_ft", "depth_min"],
    "depth_max": ["Depth max (ft)", "depth_max_ft", "Depth_max_ft", "depth_max"],
    "por_min": ["Porosity min (%)", "porosity_min_pct", "por_min"],
    "por_max": ["Porosity max (%)", "porosity_max_pct", "por_max"],
    "perm_min": ["Permeability min (mD)", "perm_min_md", "permeability_min_md", "perm_min"],
    "perm_max": ["Permeability max (mD)", "perm_max_md", "permeability_max_md", "perm_max"],
    "api_min": ["Oil gravity min (°API)", "api_min", "API min", "oil_api_min"],
    "api_max": ["Oil gravity max (°API)", "api_max", "API max", "oil_api_max"],
    "visc_min": ["Oil viscosity min (cp)", "visc_min_cp", "viscosity_min_cp", "visc_min"],
    "visc_max": ["Oil viscosity max (cp)", "visc_max_cp", "viscosity_max_cp", "visc_max"],
    "so_min": ["So at start min (%)", "so_start_min_pct", "so_min"],
    "so_max": ["So at start max (%)", "so_start_max_pct", "so_max"],
}


def normalize_col(s: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(s).strip().lower()).strip("_")


def find_column(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    exact = {str(c).strip().lower(): c for c in df.columns}
    for a in aliases:
        if a.strip().lower() in exact:
            return exact[a.strip().lower()]
    normalized = {normalize_col(c): c for c in df.columns}
    for a in aliases:
        n = normalize_col(a)
        if n in normalized:
            return normalized[n]
    return None


def discover_candidate_tables() -> List[Dict[str, Any]]:
    candidates = []
    if not ML_DATA_DIR.exists():
        raise FileNotFoundError(f"ML data directory not found: {ML_DATA_DIR}")

    for path in sorted(ML_DATA_DIR.glob("*.xlsx")):
        try:
            xls = pd.ExcelFile(path)
        except Exception as exc:
            candidates.append({"path": path, "error": str(exc)})
            continue

        for sheet in xls.sheet_names:
            try:
                df = pd.read_excel(path, sheet_name=sheet)
            except Exception as exc:
                candidates.append({
                    "path": path, "sheet": sheet, "error": str(exc)
                })
                continue

            target = find_column(df, COLUMN_ALIASES["technique"])
            formation = find_column(df, COLUMN_ALIASES["formation"])
            range_hits = sum(
                find_column(df, COLUMN_ALIASES[key]) is not None
                for key in COLUMN_ALIASES
                if key not in {"technique", "formation"}
            )

            candidates.append({
                "path": path,
                "sheet": sheet,
                "rows": len(df),
                "columns": len(df.columns),
                "target_column": target,
                "formation_column": formation,
                "range_column_hits": range_hits,
            })

    return candidates


def select_training_table(preferred_name: str = "Dataset_01_EOR.xlsx") -> Tuple[pd.DataFrame, Dict[str, Any]]:
    candidates = discover_candidate_tables()

    valid = [
        c for c in candidates
        if c.get("target_column") is not None
        and c.get("formation_column") is not None
        and c.get("range_column_hits", 0) >= 10
        and "error" not in c
    ]
    if not valid:
        raise ValueError(
            "No training table with an EOR technique target and the expected "
            "reservoir range columns was found in ml_data/."
        )

    preferred = [
        c for c in valid
        if Path(c["path"]).name.lower() == preferred_name.lower()
    ]
    chosen = preferred[0] if preferred else max(valid, key=lambda x: x["range_column_hits"])

    df = pd.read_excel(chosen["path"], sheet_name=chosen["sheet"])
    return df, chosen


def standardize_training_table(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    for target_name, aliases in COLUMN_ALIASES.items():
        source = find_column(df, aliases)
        if source is not None:
            out[target_name] = df[source]

    required = [
        "technique", "formation",
        "depth_min", "depth_max",
        "por_min", "por_max",
        "perm_min", "perm_max",
        "api_min", "api_max",
        "visc_min", "visc_max",
        "so_min", "so_max",
    ]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Training table is missing required fields: {missing}")

    for c in required:
        if c not in {"technique", "formation"}:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    out["technique"] = out["technique"].astype(str).str.strip()
    out["formation"] = (
        out["formation"].astype(str).str.strip()
        .replace({
            "Carbonate": "Carbonates",
            "Carbonates": "Carbonates",
            "Unconsolidated Sand": "Unconsolidated sands",
            "Unconsolidated sands": "Unconsolidated sands",
            "Sandstone": "Sandstone",
        })
    )

    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=required).copy()

    # Remove clearly invalid range records.
    pairs = [
        ("depth_min", "depth_max"),
        ("por_min", "por_max"),
        ("perm_min", "perm_max"),
        ("api_min", "api_max"),
        ("visc_min", "visc_max"),
        ("so_min", "so_max"),
    ]
    for lo, hi in pairs:
        out = out[out[hi] >= out[lo]]

    # Stable record identifier for duplicate checks.
    out["record_fingerprint"] = [
        hashlib.sha1(
            "|".join(str(v) for v in row).encode("utf-8")
        ).hexdigest()[:16]
        for row in out[required].itertuples(index=False, name=None)
    ]

    return out.reset_index(drop=True)


def midpoint(min_series: pd.Series, max_series: pd.Series) -> pd.Series:
    return (min_series + max_series) / 2.0


def span(min_series: pd.Series, max_series: pd.Series) -> pd.Series:
    return (max_series - min_series).clip(lower=0.0)


def build_feature_table(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    """
    Primary ML representation.

    We deliberately do NOT include:
      - '# projects' because it encodes class frequency rather than reservoir state.
      - raw_line/text identifiers.
      - fuzzy scores in the primary classifier because those may be derived from
        the same literature corpus and can create target leakage.

    Features:
      6 midpoints + 6 spans + log10(permeability) + log10(viscosity)
      + 3 formation one-hot features.
    """
    X = pd.DataFrame(index=df.index)

    X["depth_mid_ft"] = midpoint(df["depth_min"], df["depth_max"])
    X["depth_span_ft"] = span(df["depth_min"], df["depth_max"])

    X["porosity_mid_pct"] = midpoint(df["por_min"], df["por_max"])
    X["porosity_span_pct"] = span(df["por_min"], df["por_max"])

    X["perm_mid_md"] = midpoint(df["perm_min"], df["perm_max"])
    X["perm_span_md"] = span(df["perm_min"], df["perm_max"])

    X["api_mid"] = midpoint(df["api_min"], df["api_max"])
    X["api_span"] = span(df["api_min"], df["api_max"])

    X["visc_mid_cp"] = midpoint(df["visc_min"], df["visc_max"])
    X["visc_span_cp"] = span(df["visc_min"], df["visc_max"])

    X["so_mid_pct"] = midpoint(df["so_min"], df["so_max"])
    X["so_span_pct"] = span(df["so_min"], df["so_max"])

    X["log10_perm_mid"] = np.log10(np.clip(X["perm_mid_md"], 1e-6, None))
    X["log10_visc_mid"] = np.log10(np.clip(X["visc_mid_cp"], 1e-6, None))

    for formation in ["Sandstone", "Carbonates", "Unconsolidated sands"]:
        X[f"formation_{normalize_col(formation)}"] = (
            df["formation"].eq(formation).astype(float)
        )

    X = X.astype(float)
    y = df["technique"].copy()
    return X, y, list(X.columns)


def top_k_accuracy(y_true: np.ndarray, proba: np.ndarray, k: int = 3) -> float:
    pred = np.argsort(proba, axis=1)[:, -k:]
    y_true = np.asarray(y_true)
    return float(np.mean([y in row for y, row in zip(y_true, pred)]))


## 1. Discover all ML workbooks and sheets

In [6]:
candidates = discover_candidate_tables()
audit = pd.DataFrame(candidates)
display(audit)


,path,sheet,rows,columns,target_column,formation_column,range_column_hits
0,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,Montecarlo_STOIIP,5022,13,None,None,0
1,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,production,105,3,None,None,0
2,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,pressure,767,5,None,None,0
3,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,petrophysics,16,12,None,None,0
4,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,training_data,3232,11,None,Formation,0
5,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,EOR_screening,17,11,None,None,0
6,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,Montecarlo_STOIIP,5022,13,None,None,0
7,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,production,105,3,None,None,0
8,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,pressure,767,5,None,None,0
9,c:\Users\mnabielizzuddin.radz\OneDrive - PETRO...,petrophysics,16,12,None,None,0


## 2. Load the primary EOR project table

In [7]:
raw_df, chosen = select_training_table()
print('Selected:', chosen)
print('Shape:', raw_df.shape)
display(raw_df.head())


Selected: {'path': WindowsPath('c:/Users/mnabielizzuddin.radz/OneDrive - PETRONAS/Reservoir Engineering/Programming_Python_Projects/EOR ATLAS/EORWEB/EORWEBDEV/src/notebooks/ml_data/NeuroFuzzy_EOR_Extracted_Tables.xlsx'), 'sheet': 'Table1_Ranges', 'rows': 24, 'columns': 17, 'target_column': 'EOR technique', 'formation_column': 'Formation type', 'range_column_hits': 12}
Shape: (24, 17)


,EOR technique,Formation type,# projects,Depth min (ft),Depth max (ft),Porosity min (%),Porosity max (%),Permeability min (mD),Permeability max (mD),Oil gravity min (°API),Oil gravity max (°API),Oil viscosity min (cp),Oil viscosity max (cp),So at start min (%),So at start max (%),EOR production min (B/D),EOR production max (B/D)
0,Steam,Sandstone,113.0,250.0,5750.0,15.0,39.0,100.0,10000.0,8.0,22.0,18.00,500000.0,20.0,90.0,62.0,86000.0
1,Steam,Unconsolidated sands,26.0,175.0,3150.0,25.0,40.0,300.0,15000.0,9.0,25.0,175.00,200000.0,48.0,90.0,500.0,190000.0
2,Steam,Carbonates,6.0,550.0,1500.0,20.0,65.0,1.0,2000.0,10.0,29.0,26.00,4000.0,45.0,85.0,25.0,1200.0
3,Miscible CO2,Sandstone,50.0,1600.0,11950.0,10.0,28.0,9.0,2300.0,27.0,45.0,0.30,3.0,26.0,77.0,205.0,15000.0
4,Miscible CO2,Carbonates,83.0,4000.0,11100.0,4.0,24.0,0.1,5000.0,28.0,45.0,0.32,6.0,30.0,89.0,25.0,28300.0


## 3. Standardize and audit class balance

In [8]:
df = standardize_training_table(raw_df)
print('Standardized shape:', df.shape)
print('\nClass distribution:')
display(df['technique'].value_counts().rename('count').to_frame())
print('\nFormation distribution:')
display(df['formation'].value_counts().rename('count').to_frame())


Standardized shape: (17, 15)

Class distribution:


,count
technique,
Steam,3
Combustion,3
Miscible CO2,2
Miscible HC,2
Surfactants,2
Polymer,1
Nitrates,1
Microbial,1
Hot water,1



Formation distribution:


,count
formation,
Sandstone,8
Carbonates,7
Unconsolidated sands,2


In [9]:
dup_count = int(df.duplicated('record_fingerprint').sum())
class_counts = df['technique'].value_counts()
print(f'Duplicate fingerprints: {dup_count}')
print(f'Classes with < 5 records: {(class_counts < 5).sum()}')
display(df.describe(include='all').transpose())


Duplicate fingerprints: 0
Classes with < 5 records: 10


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
technique,17,10,Steam,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
formation,17,3,Sandstone,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
depth_min,17.0,NaN,NaN,NaN,2407.941176,2307.410656,175.0,550.0,1600.0,4000.0,8300.0
depth_max,17.0,NaN,NaN,NaN,6334.529412,4410.119331,1500.0,3000.0,4900.0,9500.0,14500.0
por_min,17.0,NaN,NaN,NaN,18.529412,11.716002,4.0,10.0,15.0,25.0,50.0
por_max,17.0,NaN,NaN,NaN,29.705882,13.841806,8.0,20.0,28.0,34.0,65.0
perm_min,17.0,NaN,NaN,NaN,638.541176,1934.49885,0.1,7.0,20.0,180.0,8000.0
perm_max,17.0,NaN,NaN,NaN,3757.411765,5006.310968,1.0,100.0,2000.0,5000.0,15000.0
api_min,17.0,NaN,NaN,NaN,21.694118,9.862332,8.0,12.0,23.0,30.0,37.0
api_max,17.0,NaN,NaN,NaN,32.441176,10.334835,14.0,25.0,34.0,40.0,48.0


## Interpretation
Do not move to production model selection until rare classes, duplicate records, and any near-identical project records are understood. The dataset is currently literature-derived and range-based; that is useful for research but is not equivalent to a balanced field-response dataset.